In [32]:
import pandas as pd
import numpy as np
from math import sqrt

In [33]:
movies_df = pd.read_csv(
    filepath_or_buffer='data/movies.csv'
)

ratings_df = pd.read_csv(
    filepath_or_buffer='data/ratings.csv'
)

In [34]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [35]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


In [36]:
movies_df['year'] = movies_df['title'].str.extract(pat=r'(\d{4})', expand=False)

movies_df['title'] = movies_df['title'].str.replace(pat=r'(\W{1}\d{4}\W{1})', repl='', regex=True)

movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [37]:
ratings_df.drop(
    columns=['timestamp'],
    axis=1,
    inplace=True
)

In [38]:
ratings_df.shape

(22884377, 3)

In [39]:
user_input_df = pd.DataFrame([
    {'title': 'Toy Story', 'rating': 5},
    {'title': 'Jumanji', 'rating': 5},
    {'title': 'Home', 'rating': 5},
    {'title': 'Space Jam', 'rating': 5},
    {'title': 'Who Framed Roger Rabbit?', 'rating': 5}
])

user_input_df

,title,rating
0,Toy Story,5
1,Jumanji,5
2,Home,5
3,Space Jam,5
4,Who Framed Roger Rabbit?,5


In [40]:
merged_df = pd.merge(
    right=user_input_df,
    left=movies_df,
    how='inner',
    on='title'
)

merged_df

,movieId,title,genres,year,rating
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,5
1,2,Jumanji,Adventure|Children|Fantasy,1995,5
2,673,Space Jam,Adventure|Animation|Children|Comedy|Fantasy|Sc...,1996,5
3,2987,Who Framed Roger Rabbit?,Adventure|Animation|Children|Comedy|Crime|Fant...,1988,5
4,69529,Home,Documentary,2009,5
5,79254,Home,Drama,2008,5
6,130520,Home,Adventure|Animation|Children|Comedy|Fantasy|Sc...,2015,5
7,143972,Home,(no genres listed),2013,5
8,143974,Home,Drama,2011,5


In [41]:
merged_df.drop(
    index=[4, 5, 7, 8],
    axis=0,
    inplace=True
)

merged_df

,movieId,title,genres,year,rating
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,5
1,2,Jumanji,Adventure|Children|Fantasy,1995,5
2,673,Space Jam,Adventure|Animation|Children|Comedy|Fantasy|Sc...,1996,5
3,2987,Who Framed Roger Rabbit?,Adventure|Animation|Children|Comedy|Crime|Fant...,1988,5
6,130520,Home,Adventure|Animation|Children|Comedy|Fantasy|Sc...,2015,5


In [42]:
user_subset_df = pd.merge(
    left=ratings_df,
    right=merged_df,
    how='inner',
    on='movieId'
)

print(f'Shape: {user_subset_df.shape}')
user_subset_df.head()

Shape: (119049, 7)


,userId,movieId,rating_x,title,genres,year,rating_y
0,13,2,2.0,Jumanji,Adventure|Children|Fantasy,1995,5
1,13,673,2.0,Space Jam,Adventure|Animation|Children|Comedy|Fantasy|Sc...,1996,5
2,15,1,4.0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,5
3,15,2987,3.5,Who Framed Roger Rabbit?,Adventure|Animation|Children|Comedy|Crime|Fant...,1988,5
4,17,1,5.0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,5


In [43]:
user_subset_df.drop(
    columns=['title', 'rating_y'],
    axis=1,
    inplace=True
)

user_subset_df.rename(
    columns={
        'rating_x': 'rating'
    },
    inplace=True
)

user_subset_df.head()

,userId,movieId,rating,genres,year
0,13,2,2.0,Adventure|Children|Fantasy,1995
1,13,673,2.0,Adventure|Animation|Children|Comedy|Fantasy|Sc...,1996
2,15,1,4.0,Adventure|Animation|Children|Comedy|Fantasy,1995
3,15,2987,3.5,Adventure|Animation|Children|Comedy|Crime|Fant...,1988
4,17,1,5.0,Adventure|Animation|Children|Comedy|Fantasy,1995


In [44]:
user_subset_group = user_subset_df.groupby(by='userId')
for user, group in user_subset_group:
    print(
        f'====================\n'
        f'User: {user}\n'
        f'Group:\n'
        f'{group}'
    )
    break

User: 13
Group:
   userId  movieId  rating                                             genres  \
0      13        2     2.0                         Adventure|Children|Fantasy   
1      13      673     2.0  Adventure|Animation|Children|Comedy|Fantasy|Sc...   

   year  
0  1995  
1  1996  


In [45]:
sorted_user_subset_group = sorted(
    user_subset_group,
    key=lambda x: len(x[1]),
    reverse=True
)

sorted_user_subset_group

[(4208,
        userId  movieId  rating  \
  2017    4208        1     4.0   
  2018    4208        2     2.0   
  2019    4208      673     2.0   
  2020    4208     2987     4.0   
  2021    4208   130520     3.0   
  
                                                   genres  year  
  2017        Adventure|Animation|Children|Comedy|Fantasy  1995  
  2018                         Adventure|Children|Fantasy  1995  
  2019  Adventure|Animation|Children|Comedy|Fantasy|Sc...  1996  
  2020  Adventure|Animation|Children|Comedy|Crime|Fant...  1988  
  2021  Adventure|Animation|Children|Comedy|Fantasy|Sc...  2015  ),
 (13614,
        userId  movieId  rating  \
  6544   13614        1     5.0   
  6545   13614        2     3.0   
  6546   13614      673     2.0   
  6547   13614     2987     3.0   
  6548   13614   130520     4.0   
  
                                                   genres  year  
  6544        Adventure|Animation|Children|Comedy|Fantasy  1995  
  6545                     

In [46]:
pearson_correlation_dict = {}

merged_base = merged_df[['movieId', 'rating']].dropna().copy()
merged_base = merged_base.sort_values('movieId')

for userId, group in sorted_user_subset_group:
    grp = group[['movieId', 'rating']].dropna().copy()
    grp = grp.sort_values('movieId')

    aligned = merged_base.merge(grp, on='movieId', how='inner', suffixes=('_base', '_grp'))

    n = len(aligned)
    if n < 2:
        pearson_correlation_dict[userId] = 0.0
        continue

    x = aligned['rating_base'].to_numpy(dtype=float)
    y = aligned['rating_grp'].to_numpy(dtype=float)

    # Pearson (el ile, stabil)
    x_mean = x.mean()
    y_mean = y.mean()

    dx = x - x_mean
    dy = y - y_mean

    denom = np.sqrt((dx * dx).sum() * (dy * dy).sum())
    if denom == 0:
        pearson_correlation_dict[userId] = 0.0
    else:
        pearson_correlation_dict[userId] = float((dx * dy).sum() / denom)


In [47]:
print(pearson_correlation_dict)

{4208: 0.0, 13614: 0.0, 26125: 0.0, 33902: 0.0, 36946: 0.0, 39142: 0.0, 40145: 0.0, 44027: 0.0, 45221: 0.0, 47529: 0.0, 47792: 0.0, 53735: 0.0, 58040: 0.0, 76703: 0.0, 88296: 0.0, 96736: 0.0, 99992: 0.0, 100668: 0.0, 101410: 0.0, 117070: 0.0, 121742: 0.0, 129608: 0.0, 139796: 0.0, 140615: 0.0, 151549: 0.0, 153902: 0.0, 157267: 0.0, 159251: 0.0, 162517: 0.0, 168524: 0.0, 172955: 0.0, 180362: 0.0, 195621: 0.0, 195637: 0.0, 199909: 0.0, 206700: 0.0, 226555: 0.0, 237434: 0.0, 243600: 0.0, 245940: 0.0, 17: 0.0, 114: 0.0, 277: 0.0, 393: 0.0, 407: 0.0, 479: 0.0, 558: 0.0, 675: 0.0, 815: 0.0, 1130: 0.0, 1204: 0.0, 1343: 0.0, 1502: 0.0, 1513: 0.0, 1573: 0.0, 1599: 0.0, 1625: 0.0, 1824: 0.0, 1858: 0.0, 1950: 0.0, 1992: 0.0, 2065: 0.0, 2106: 0.0, 2115: 0.0, 2204: 0.0, 2372: 0.0, 2397: 0.0, 2429: 0.0, 2569: 0.0, 2596: 0.0, 2625: 0.0, 2791: 0.0, 2814: 0.0, 2839: 0.0, 2964: 0.0, 3127: 0.0, 3281: 0.0, 3358: 0.0, 3388: 0.0, 3429: 0.0, 3596: 0.0, 3734: 0.0, 4099: 0.0, 4415: 0.0, 4458: 0.0, 4576: 0.0, 4